# Notebook 09: Memory Profiling and Debugging

## Why This Matters

Out-of-memory errors are the most common practical blocker when scaling models. But OOM is not just an inconvenience -- if you don't understand *why* memory was exhausted, you will fix it with a smaller batch size (hurting training dynamics) when the real fix is avoiding activation accumulation in eval mode, using gradient checkpointing, or fixing a memory leak in a training loop. This notebook gives you the mental model and tools to diagnose and fix memory issues systematically rather than by trial and error.

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
print(f"torch: {torch.__version__}")
print("Ready.")

## 1. CUDA Memory Concepts -- allocated vs reserved vs active

PyTorch's CUDA memory allocator maintains a **cache** (reserved pool) to avoid frequent calls to `cudaMalloc`/`cudaFree` which are expensive. Three key metrics:

- **allocated**: memory currently occupied by live tensors
- **reserved**: memory held by PyTorch's allocator (may be larger than allocated due to caching)
- **active**: memory in the current computation graph (allocated + about to be freed)

On non-CUDA devices we measure CPU memory as a proxy. The patterns and debugging approaches are identical.

In [ ]:
def memory_stats():
    if device == "cuda":
        allocated = torch.cuda.memory_allocated(device) / 1e6
        reserved  = torch.cuda.memory_reserved(device)  / 1e6
        print(f"  allocated: {allocated:.1f} MB")
        print(f"  reserved:  {reserved:.1f} MB")
    else:
        # CPU: count live tensors as proxy
        import gc
        live = sum(1 for obj in gc.get_objects() if isinstance(obj, torch.Tensor))
        print(f"  (CPU device: {live} live tensors as proxy for GPU memory)")

# Demonstrate memory lifecycle
print("Before allocating:")
memory_stats()

tensors = [torch.randn(1000, 1000, device=device) for _ in range(5)]
print("\nAfter allocating 5 x (1000x1000):")
memory_stats()

del tensors
if device == "cuda":
    torch.cuda.empty_cache()
print("\nAfter del + empty_cache:")
memory_stats()

# What eats memory in a training step:
# 1. Parameters: model weights
# 2. Optimizer states: Adam has 2 tensors per parameter (m and v)
# 3. Activations: all intermediate values kept for backward()
# 4. Gradients: same shape as parameters
model_demo = nn.Sequential(nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, 512))
param_bytes = sum(p.numel() * p.element_size() for p in model_demo.parameters())
print(f"\nModel parameter memory:  {param_bytes/1e6:.2f} MB")
print(f"Optimizer state (Adam):  ~{2 * param_bytes/1e6:.2f} MB (m + v)")
print(f"Gradient memory:         ~{param_bytes/1e6:.2f} MB")
print(f"Total optimizer overhead: ~{4 * param_bytes/1e6:.2f} MB per model copy")

## 2. Activation Accumulation -- The Silent OOM in Eval Mode

The most common cause of OOM during evaluation: calling the model in a loop without `torch.no_grad()`. Every forward pass builds the computation graph and stores all intermediate activations. After 1000 batches, you have 1000 graphs in memory. This is not a hypothetical edge case -- it happens every time a junior engineer runs evaluation without no_grad.

In [ ]:
import gc

def simulate_memory_leak_eval(model, loader_size=20):
    # WRONG: no torch.no_grad() -- activations accumulate
    all_logits = []
    for i in range(loader_size):
        x = torch.randn(32, 16, device=device)
        logits = model(x)  # builds graph, stores activations
        all_logits.append(logits)  # keeps reference, can't be freed
    return torch.cat(all_logits)

def simulate_correct_eval(model, loader_size=20):
    # CORRECT: no_grad prevents graph construction
    all_logits = []
    with torch.no_grad():
        for i in range(loader_size):
            x = torch.randn(32, 16, device=device)
            logits = model(x)  # no graph, activations freed after each step
            all_logits.append(logits.detach())
    return torch.cat(all_logits)

model_eval = nn.Sequential(nn.Linear(16, 64), nn.ReLU(), nn.Linear(64, 5))

# Count graph nodes as proxy for memory
import gc
gc.collect()
n_before = sum(1 for o in gc.get_objects() if isinstance(o, torch.Tensor))
_ = simulate_memory_leak_eval(model_eval)
gc.collect()
n_leak = sum(1 for o in gc.get_objects() if isinstance(o, torch.Tensor))

gc.collect()
n_before2 = sum(1 for o in gc.get_objects() if isinstance(o, torch.Tensor))
_ = simulate_correct_eval(model_eval)
gc.collect()
n_correct = sum(1 for o in gc.get_objects() if isinstance(o, torch.Tensor))

print(f"Without no_grad: +{n_leak - n_before} tensors remain after eval")
print(f"With no_grad:    +{n_correct - n_before2} tensors remain after eval")
print("\nAlways use torch.no_grad() (or @torch.inference_mode()) in eval loops.")

## 3. Gradient Checkpointing

During the forward pass PyTorch stores ALL intermediate activations needed for the backward pass. For a deep model this can be 10-20x more memory than the parameters alone. Gradient checkpointing (activation checkpointing) trades compute for memory: it does NOT store intermediate activations during the forward pass, and instead recomputes them on-demand during the backward pass. This reduces activation memory from O(layers) to O(sqrt(layers)) at the cost of ~33% more FLOPs.

In [ ]:
from torch.utils.checkpoint import checkpoint

class DeepMLP(nn.Module):
    def __init__(self, n_layers=8, width=256):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(width, width), nn.GELU())
            for _ in range(n_layers)
        ])
        self.out = nn.Linear(width, 10)

    def forward_eager(self, x):
        # Normal forward: stores all activations
        for layer in self.layers:
            x = layer(x)
        return self.out(x)

    def forward_checkpointed(self, x):
        # Checkpointed: recomputes activations during backward
        for layer in self.layers:
            x = checkpoint(layer, x, use_reentrant=False)
        return self.out(x)

    def forward(self, x):
        return self.forward_checkpointed(x)

model_ckpt = DeepMLP(n_layers=8, width=256)
x_ck = torch.randn(32, 256, requires_grad=True)

# Eager: count activations retained in graph
loss_eager = model_ckpt.forward_eager(x_ck).sum()
eager_tensors_in_graph = sum(1 for o in gc.get_objects()
                              if isinstance(o, torch.Tensor) and o.requires_grad)
del loss_eager; gc.collect()

# Checkpointed
loss_ckpt = model_ckpt.forward_checkpointed(x_ck).sum()
ckpt_tensors_in_graph = sum(1 for o in gc.get_objects()
                             if isinstance(o, torch.Tensor) and o.requires_grad)
loss_ckpt.backward()
del loss_ckpt; gc.collect()

print(f"Eager:       ~{eager_tensors_in_graph} grad tensors in graph")
print(f"Checkpointed: ~{ckpt_tensors_in_graph} grad tensors in graph (lower = less memory)")
print("\nGradient checkpointing reduces activation memory at cost of ~33% more compute.")
print("Use for: large models, long sequences, video/3D data where activations dominate.")

## 4. torch.autograd.set_detect_anomaly -- NaN/Inf Hunting

`detect_anomaly` mode makes autograd raise an exception with a Python stack trace when it encounters the first `NaN` or `Inf` in a gradient. Without it, NaN propagates silently through the entire backward pass and you only notice it in the loss curve. The overhead is significant (2-5x slower), so use it only for debugging, never in production.

In [ ]:
# Simulate a NaN-producing bug
def buggy_forward(x):
    # log(0) = -inf, then multiplied -> NaN gradient
    return torch.log(x).sum()

x_nan = torch.tensor([1.0, 0.0, 2.0], requires_grad=True)

print("Without detect_anomaly:")
loss = buggy_forward(x_nan)
try:
    loss.backward()
    print(f"  x_nan.grad: {x_nan.grad}  (NaN present, no exception)")
except Exception as e:
    print(f"  Error: {e}")

x_nan2 = torch.tensor([1.0, 0.0, 2.0], requires_grad=True)
print("\nWith detect_anomaly:")
try:
    with torch.autograd.set_detect_anomaly(True):
        loss2 = buggy_forward(x_nan2)
        loss2.backward()
except RuntimeError as e:
    print(f"  Caught NaN at source: {str(e)[:200]}")

# The fix: clamp inputs to avoid log(0)
x_fixed = torch.tensor([1.0, 0.0, 2.0], requires_grad=True)
loss_fixed = torch.log(x_fixed.clamp(min=1e-8)).sum()
loss_fixed.backward()
print(f"\nWith clamp fix: x.grad = {x_fixed.grad}")

## 5. Loss Diagnostic -- What CrossEntropyLoss Should Output at Init

A sanity check every training run should include: at random initialisation, the loss from `CrossEntropyLoss` for K classes should be approximately $\log(K)$. If the initial loss is wildly different, your model architecture, weight initialisation, or data preprocessing has a bug -- before you even start training.

In [ ]:
# Sanity check: initial loss should be ~log(K)
import math

for K in [2, 5, 10, 100, 1000]:
    # Random init model
    torch.manual_seed(42)
    model_sanity = nn.Sequential(nn.Linear(32, K))
    x_sanity = torch.randn(1000, 32)
    y_sanity = torch.randint(0, K, (1000,))

    with torch.no_grad():
        logits = model_sanity(x_sanity)
        loss = F.cross_entropy(logits, y_sanity)

    expected = math.log(K)
    delta = abs(loss.item() - expected)
    status = "OK" if delta < 0.5 else "SUSPICIOUS"
    print(f"K={K:5d}: loss={loss.item():.4f}  expected={expected:.4f}  diff={delta:.4f}  [{status}]")

print("\nIf initial loss >> log(K): model outputs are not uniform (bad init or BN issue)")
print("If initial loss << log(K): labels might be leaking into features (data bug)")

## 6. torch.profiler -- Compute-Bound vs Memory-Bound vs Launch-Bound

Profiling tells you WHERE your training time is going. The three regimes:
- **Compute-bound**: GPU ALUs are the bottleneck. Solution: fewer or more efficient computations (fusion, quantisation).
- **Memory-bound**: data movement between HBM and ALUs dominates. Solution: kernel fusion, tiling, reduce memory reads.
- **Launch-bound**: CPU is saturated dispatching kernel launches. Solution: `torch.compile(mode="reduce-overhead")`, operator fusion.

In [ ]:
from torch.profiler import profile, record_function, ProfilerActivity

model_prof = nn.Sequential(
    nn.Linear(256, 1024), nn.ReLU(),
    nn.Linear(1024, 1024), nn.ReLU(),
    nn.Linear(1024, 256), nn.ReLU(),
    nn.Linear(256, 10),
).to(device)

x_prof = torch.randn(64, 256, device=device)
y_prof = torch.randint(0, 10, (64,), device=device)
optimizer_prof = torch.optim.AdamW(model_prof.parameters(), lr=1e-3)

activities = [ProfilerActivity.CPU]
if device == "cuda":
    activities.append(ProfilerActivity.CUDA)

with profile(
    activities=activities,
    record_shapes=True,
    with_flops=True,
    profile_memory=True,
) as prof:
    for step in range(3):
        with record_function(f"step_{step}"):
            optimizer_prof.zero_grad(set_to_none=True)
            with record_function("forward"):
                loss = F.cross_entropy(model_prof(x_prof), y_prof)
            with record_function("backward"):
                loss.backward()
            with record_function("optimizer_step"):
                optimizer_prof.step()

# Print top ops by CPU time
print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=10))

## Summary

### Key Concepts

| Concept | Key Point |
|---------|----------|
| allocated vs reserved | Allocated = live tensors; reserved = allocator cache (can exceed allocated) |
| Eval without no_grad | Every forward builds a graph; 1000 batches = 1000 graphs in memory |
| Gradient checkpointing | Don't store activations; recompute in backward; trades compute for memory |
| detect_anomaly | Raises exception at first NaN/Inf with stack trace; debugging only |
| Initial loss = log(K) | Sanity check at random init; deviation indicates architecture/data bug |
| torch.profiler | Identifies compute-bound, memory-bound, or launch-bound bottlenecks |

### Common Pitfalls
- Running eval loop without `torch.no_grad()` -- quadratic memory growth
- Using gradient checkpointing on a small model -- overhead exceeds savings
- Ignoring `detect_anomaly` overhead in production -- 5x slowdown
- Empty_cache() instead of fixing the leak -- frees cache but not live tensors
- Not checking initial loss -- trains for 10 epochs before noticing the model is broken

### Next: Notebook 10 -- torch.func Transforms